# 05 — Ocean Data: GODAS and OISST

Plot oceanic fields from two NOAA sources entirely from scratch:

| Source | Variables | Resolution | Period |
|---|---|---|---|
| **NOAA OISST v2 High-Res** | SST | 0.25° | 1981–present |
| **NOAA NCEP GODAS** | pottmp, salt, ucur, vcur, sshg | ~1°×⅓°, 40 levels | 1980–present |

Both are accessed via **noawclg** (which wraps OPeNDAP — no manual downloads).

---
**Stack used:** `noawclg`, `xarray`, `numpy`, `matplotlib`, `cartopy`, `cmocean`, `scipy`

In [ ]:
import noawclg
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmocean
import copy
from scipy.interpolate import RegularGridInterpolator

In [ ]:
BG = "#0d1117"; LAND = "#13171d"; OCEAN_BG = "#0d1520"
COAST = "#2d6db5"; BORDER = "#2a2f36"
TEXT = "#e6edf3"; SUBTEXT = "#8b949e"
BOXBG = "#161b22"; BOXEDGE = "#30363d"

plt.rcParams.update({
    "figure.facecolor":  BG,
    "axes.facecolor":    BG,
    "savefig.facecolor": BG,
    "text.color":        TEXT,
    "font.family":       "monospace",
})

## Part A — OISST v2 Sea Surface Temperature

OISST is a daily/monthly 0.25° SST product. noawclg wraps the OPeNDAP endpoint at NOAA PSL.

### A1. Load OISST via noawclg

In [ ]:
# Monthly mean SST — May 2024
YEAR  = 2024
MONTH = 5

lat_sst, lon_sst, sst = noawclg.open_oisst(YEAR, MONTH)
# Returns:
#   lat_sst  — 1D array, −90…90 at 0.25° spacing
#   lon_sst  — 1D array, −180…180 at 0.25° spacing (already converted)
#   sst      — 2D array (lat, lon), °C, NaN over land

print(f"SST shape : {sst.shape}")
print(f"Lat range : {lat_sst[0]:.2f} … {lat_sst[-1]:.2f}")
print(f"Lon range : {lon_sst[0]:.2f} … {lon_sst[-1]:.2f}")
print(f"SST range : {np.nanmin(sst):.1f} … {np.nanmax(sst):.1f} °C")

### A2. Load OISST directly from OPeNDAP (without noawclg)

If you prefer to skip noawclg entirely, here is the raw xarray approach:

In [ ]:
OPENDAP_URL = (
    "https://psl.noaa.gov/thredds/dodsC/"
    "Datasets/noaa.oisst.v2.highres/sst.mnmean.nc"
)

ds_oisst = xr.open_dataset(OPENDAP_URL, engine="netcdf4")

# Select the target month (time is monthly)
target = ds_oisst.sel(time=f"{YEAR}-{MONTH:02d}", method="nearest")
sst_raw = target["sst"].squeeze().values      # (lat, lon), °C
lat_raw = ds_oisst["lat"].values               # 0.25° spacing
lon_raw = ds_oisst["lon"].values               # 0–360

# Convert 0–360 → −180/180
lon_180 = np.where(lon_raw > 180, lon_raw - 360, lon_raw)
order   = np.argsort(lon_180)
lon_180 = lon_180[order]
sst_raw = sst_raw[:, order]

# Mask fill values
sst_raw = np.where(sst_raw > 100, np.nan, sst_raw)

print("OISST direct load — shape:", sst_raw.shape)

### A3. Bilinear upsampling (optional but recommended for GODAS)

OISST at 0.25° is already smooth. GODAS at ~1° looks blocky — upsample it 4× for a clean result.

In [ ]:
def upsample(lat, lon, field, factor=4):
    """Bilinear upsampling via RegularGridInterpolator. NaN propagates."""
    interp = RegularGridInterpolator(
        (lat, lon), field,
        method="linear",
        bounds_error=False,
        fill_value=np.nan,
    )
    lat_fine = np.linspace(lat[0],  lat[-1],  len(lat)  * factor)
    lon_fine = np.linspace(lon[0],  lon[-1],  len(lon)  * factor)
    lon_g, lat_g = np.meshgrid(lon_fine, lat_fine)
    return lat_fine, lon_fine, interp((lat_g, lon_g))

### A4. Plot SST on an orthographic globe

In [ ]:
# Use the noawclg-loaded data (already in −180/180)
# If you used the direct load above, replace with lat_raw/lon_180/sst_raw
plot_lat, plot_lon, plot_field = lat_sst, lon_sst, sst

levels_sst = np.arange(-2, 34, 2)
cmap_sst   = copy.copy(cmocean.cm.thermal)
cmap_sst.set_under(alpha=0); cmap_sst.set_bad(alpha=0)
norm_sst   = mcolors.BoundaryNorm(levels_sst, ncolors=cmap_sst.N, clip=False)

proj = ccrs.Orthographic(central_longitude=0, central_latitude=0)

fig, ax = plt.subplots(figsize=(10, 10),
                        subplot_kw={"projection": proj}, facecolor=BG)
ax.set_facecolor(BG)
ax.set_global()

ax.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,     zorder=2)
ax.add_feature(cfeature.COASTLINE.with_scale("110m"),
               edgecolor=COAST, linewidth=0.6, zorder=3)
ax.add_feature(cfeature.BORDERS.with_scale("110m"),
               edgecolor=BORDER, linewidth=0.3, zorder=3)

cf = ax.pcolormesh(
    plot_lon, plot_lat, plot_field,
    cmap=cmap_sst, norm=norm_sst,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

cb = fig.colorbar(cf, ax=ax, orientation="horizontal",
                  pad=0.03, fraction=0.03, shrink=0.85)
cb.set_label("SST (°C)", fontsize=8, color=SUBTEXT)
cb.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb.outline.set_edgecolor(BOXEDGE)

ax.set_title("Sea Surface Temperature (OISST v2 0.25°)",
             fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax.set_title(f"{YEAR}-{MONTH:02d}",
             fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig.tight_layout()
fig.savefig("oisst_ortho.png", dpi=150, bbox_inches="tight")
plt.show()

---
## Part B — GODAS Subsurface Fields

### B1. Load GODAS potential temperature at a given depth

In [ ]:
# Available GODAS variables: pottmp, salt, ucur, vcur, sshg
DEPTH_M = 200   # metres — GODAS level closest to this value

lat_g, lon_g_raw, pottmp = noawclg.get_godas(
    var="pottmp",
    year=YEAR,
    month=MONTH,
    depth_m=DEPTH_M,
)
# pottmp is in Kelvin — convert to °C
pottmp_c = pottmp - 273.15

# GODAS lon is 0–360 → −180/180
lon_g_180 = np.where(lon_g_raw > 180, lon_g_raw - 360, lon_g_raw)
order_g   = np.argsort(lon_g_180)
lon_g_180 = lon_g_180[order_g]
pottmp_c  = pottmp_c[:, order_g]

print(f"GODAS pottmp@{DEPTH_M}m — shape: {pottmp_c.shape}")
print(f"Range: {np.nanmin(pottmp_c):.1f} … {np.nanmax(pottmp_c):.1f} °C")

### B2. Upsample the coarse GODAS grid

In [ ]:
lat_fine, lon_fine, pottmp_fine = upsample(lat_g, lon_g_180, pottmp_c, factor=4)
print(f"Upsampled shape: {pottmp_fine.shape}")

### B3. Plot on a PlateCarrée tropical belt

In [ ]:
levels_pt = np.arange(-2, 32, 2)
cmap_pt   = copy.copy(cmocean.cm.thermal)
cmap_pt.set_under(alpha=0); cmap_pt.set_bad(alpha=0)
norm_pt   = mcolors.BoundaryNorm(levels_pt, ncolors=cmap_pt.N, clip=False)

# Tropical belt extent
W, E, S, N = -180, 180, -30, 30

proj_pc = ccrs.PlateCarree()
fig2, ax2 = plt.subplots(
    figsize=(16, 5),
    subplot_kw={"projection": proj_pc},
    facecolor=BG,
)
ax2.set_facecolor(BG)
ax2.set_extent([W, E, S, N], crs=ccrs.PlateCarree())

ax2.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax2.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.6, zorder=3)
ax2.add_feature(cfeature.BORDERS.with_scale("110m"),
                edgecolor=BORDER, linewidth=0.3, zorder=3)

cf2 = ax2.pcolormesh(
    lon_fine, lat_fine, pottmp_fine,
    cmap=cmap_pt, norm=norm_pt,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

cb2 = fig2.colorbar(cf2, ax=ax2, orientation="vertical",
                    pad=0.02, fraction=0.015, shrink=0.9)
cb2.set_label(f"Potential Temperature @ {DEPTH_M} m (°C)",
              fontsize=8, color=SUBTEXT)
cb2.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb2.outline.set_edgecolor(BOXEDGE)

ax2.set_title(f"GODAS Potential Temperature — {DEPTH_M} m depth",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax2.set_title(f"{YEAR}-{MONTH:02d}  |  NOAA NCEP GODAS",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig2.tight_layout()
fig2.savefig(f"godas_pottmp_{DEPTH_M}m_tropical.png", dpi=150, bbox_inches="tight")
plt.show()

### B4. GODAS Sea Surface Height

In [ ]:
lat_g2, lon_g2_raw, sshg = noawclg.get_godas(
    var="sshg",
    year=YEAR,
    month=MONTH,
    depth_m=None,   # sshg has no depth dimension
)

lon_g2_180 = np.where(lon_g2_raw > 180, lon_g2_raw - 360, lon_g2_raw)
order_g2   = np.argsort(lon_g2_180)
lon_g2_180 = lon_g2_180[order_g2]
sshg       = sshg[:, order_g2]

lat_f2, lon_f2, sshg_fine = upsample(lat_g2, lon_g2_180, sshg, factor=4)

levels_ssh = np.arange(-0.5, 0.55, 0.05)
cmap_ssh   = copy.copy(cmocean.cm.balance)
cmap_ssh.set_under(alpha=0); cmap_ssh.set_bad(alpha=0)
norm_ssh   = mcolors.BoundaryNorm(levels_ssh, ncolors=cmap_ssh.N, clip=False)

proj_p = ccrs.PlateCarree()
fig3, ax3 = plt.subplots(
    figsize=(16, 8),
    subplot_kw={"projection": proj_p},
    facecolor=BG,
)
ax3.set_facecolor(BG)
ax3.set_global()

ax3.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax3.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.5, zorder=3)

cf3 = ax3.pcolormesh(
    lon_f2, lat_f2, sshg_fine,
    cmap=cmap_ssh, norm=norm_ssh,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

cb3 = fig3.colorbar(cf3, ax=ax3, orientation="horizontal",
                    pad=0.04, fraction=0.025, shrink=0.7)
cb3.set_label("SSH anomaly (m)", fontsize=8, color=SUBTEXT)
cb3.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb3.outline.set_edgecolor(BOXEDGE)

ax3.set_title("Sea Surface Height relative to geoid (GODAS)",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax3.set_title(f"{YEAR}-{MONTH:02d}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig3.tight_layout()
fig3.savefig("godas_sshg_global.png", dpi=150, bbox_inches="tight")
plt.show()

### B5. Ocean current vectors (ucur + vcur)

In [ ]:
lat_u, lon_u_raw, ucur = noawclg.get_godas("ucur", YEAR, MONTH, depth_m=5)
lat_v, lon_v_raw, vcur = noawclg.get_godas("vcur", YEAR, MONTH, depth_m=5)

lon_u_180 = np.where(lon_u_raw > 180, lon_u_raw - 360, lon_u_raw)
order_u   = np.argsort(lon_u_180)
lon_u_180 = lon_u_180[order_u]
ucur      = ucur[:, order_u]
vcur      = vcur[:, order_u]

# Current speed
speed = np.sqrt(ucur**2 + vcur**2)

levels_spd = np.arange(0, 2.6, 0.1)
cmap_spd   = copy.copy(cmocean.cm.speed)
cmap_spd.set_under(alpha=0); cmap_spd.set_bad(alpha=0)
norm_spd   = mcolors.BoundaryNorm(levels_spd, ncolors=cmap_spd.N, clip=False)

W, E, S, N = -180, 180, -60, 60
proj_cur   = ccrs.PlateCarree()

fig4, ax4 = plt.subplots(
    figsize=(16, 7),
    subplot_kw={"projection": proj_cur},
    facecolor=BG,
)
ax4.set_facecolor(BG)
ax4.set_extent([W, E, S, N], crs=ccrs.PlateCarree())

ax4.add_feature(cfeature.LAND.with_scale("110m"),     facecolor=LAND,  zorder=2)
ax4.add_feature(cfeature.COASTLINE.with_scale("110m"),
                edgecolor=COAST, linewidth=0.5, zorder=3)

# Speed background
cf4 = ax4.pcolormesh(
    lon_u_180, lat_u, speed,
    cmap=cmap_spd, norm=norm_spd,
    transform=ccrs.PlateCarree(),
    zorder=1,
)

# Current arrows — subsample every 5th point to avoid overcrowding
step = 5
ax4.quiver(
    lon_u_180[::step], lat_u[::step],
    ucur[::step, ::step], vcur[::step, ::step],
    transform=ccrs.PlateCarree(),
    scale=20, width=0.002,
    color="#e6edf3", alpha=0.55,
    zorder=4,
)

cb4 = fig4.colorbar(cf4, ax=ax4, orientation="horizontal",
                    pad=0.04, fraction=0.025, shrink=0.7)
cb4.set_label("Current Speed @ 5 m (m s⁻¹)", fontsize=8, color=SUBTEXT)
cb4.ax.tick_params(labelsize=7, colors=SUBTEXT)
cb4.outline.set_edgecolor(BOXEDGE)

ax4.set_title("Ocean Surface Currents (GODAS, 5 m depth)",
              fontsize=10, fontweight="bold", color=TEXT, loc="left", pad=6)
ax4.set_title(f"{YEAR}-{MONTH:02d}",
              fontsize=7, color=SUBTEXT, loc="right", pad=6)

fig4.tight_layout()
fig4.savefig("godas_currents_5m.png", dpi=150, bbox_inches="tight")
plt.show()

## Exercises

1. **Depth profile** — loop over `[5, 50, 100, 200, 500]` m and save a separate GODAS pottmp plot for each depth.
2. **El Niño region** — crop to `lon=[-180, -70]`, `lat=[-10, 10]` and compute the Niño 3.4 area mean: `np.nanmean(sst[np.abs(lat_sst)<=5, :])` after slicing.
3. **Salinity** — load `"salt"` (kg/kg → PSU: ×1000), use `cmocean.cm.haline`.
4. **Anomaly map** — load the same month for two different years, subtract the fields, and plot with `cmocean.cm.balance`.
5. **Pacific-centred globe** — use `ccrs.Orthographic(central_longitude=200, central_latitude=0)` and rerun any plot above.